# Topic modelling 

This workbook implements topic modelling. 

It first pre-processes the text. This first consists of performing named entity recognition (NER) - utilising the pre-trained HuggingFace model ([reddit-ner-place-names](https://huggingface.co/cjber/reddit-ner-place_names) developed by Cillian Berragan at Liverpool uni. Then, I remove stop words, punctuation and lemmatize the text. 

Topic modelling is carried out using various methods and compared. This includes:
* Latent dirichlet allocation (LDA)
* Latent semantic analysis (LSA)
* bertopic 

In [ ]:
import pandas as pd
import numpy as np
import re
import string 
from datasets import Dataset

from sklearn.feature_extraction.text import CountVectorizer
from scipy.cluster import hierarchy as sch

from transformers import AutoModelForMaskedLM, pipeline

from bertopic import BERTopic

import nltk 
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from gensim import corpora
from gensim.models import LdaModel
from gensim.models import LsiModel
from gensim.models import HdpModel
from gensim.models import TfidfModel

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to /Users/bea/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/bea/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/bea/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Start by loading the data - this is a small chunk of data which has been scraped. 

In [2]:
# Load data
train_df = pd.read_csv('../outputs/train_comments.csv')
test_df = pd.read_csv('../outputs/test_comments.csv')

# Convert DataFrame to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [3]:
print(f'Number of comments in training dataset: {len(train_dataset)} \nNumber of comments in test dataset: {len(test_dataset)}')

Number of comments in training dataset: 542 
Number of comments in test dataset: 136


In [4]:
text = list(train_df['comment_text'])

## Baseline topic modelling acheived with BertTopic and minimal pre-processing 

This uses the pretrained domain knowledge model which was an output of 1_0_1_domain_knowledge_nlp.ipynb

In [5]:
model = AutoModelForMaskedLM.from_pretrained("../outputs/nlp_fine_tuning/distilbert-base-uncased")

In [6]:
# Some generic pre-processing steps for the data
text = [sentence.split('.') for sentence in text]
text = [item for sublist in text for item in sublist]
text = [sentence.replace('\n', '') for sentence in text]

In [7]:
topic_model = BERTopic(embedding_model=model)
topics, probs = topic_model.fit_transform(text)

topic_model.get_topic_info()[:10]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1203,-1_the_and_in_of,"[the, and, in, of, to, this, as, is, with, area]",[I object to this application as I live by the...
1,0,450,0_amazing_https_agree_better,"[amazing, https, agree, better, see, do, it, ,...","[, do better, See: https://m]"
2,1,168,1_green_space_was_spaces,"[green, space, was, spaces, more, land, area, ...",[This space was promised by the developer as a...
3,2,143,2_community_space_groups_use,"[community, space, groups, use, reduction, for...",[ It appears that there will be less than 10% ...
4,3,137,3_school_nursery_next_children,"[school, nursery, next, children, primary, cla...",[ This is directly next to a school and the du...
5,4,124,4_forest_gate_woodgrange_community,"[forest, gate, woodgrange, community, road, ce...","[Forest Gate needs more community spaces, not ..."
6,5,113,5_hall_durning_community_groups,"[hall, durning, community, groups, we, at, act...","[The building which is being lost, Durning Hal..."
7,6,98,6_parking_car_already_spaces,"[parking, car, already, spaces, insufficient, ...",[ The proposed provision of car parking spaces...
8,7,95,7_housing_affordable_residential_westminster,"[housing, affordable, residential, westminster...",[Please Westminster City Council and The UK Go...
9,8,76,8_sunlight_daylight_overshadowing_block,"[sunlight, daylight, overshadowing, block, my,...","[Access to daylight:Daylight, Sunlight & Overs..."


## Removing geographic place names using named entity recognition 
I don't want the topics identified to relate to the place names of specific applications (i.e. Durning Hall or Forest Gate) - as I want the topics to be generalised themes, common across applications - hence I remove all place names. This uses Named Entity Recognition (NER), I intially tried using the out of the box bog-standard model, but it wasn't able to recognise more specific British place names. Instead I use the "cjber/reddit-ner-place_names" - which has specifically been trained to recognise these sorts of place names. 

In [8]:
ner_pipeline = pipeline(
    task="ner",
    model="cjber/reddit-ner-place_names",
    tokenizer="cjber/reddit-ner-place_names",
    aggregation_strategy="first",
)

Device set to use mps:0


In [9]:
def remove_locations(text):
    ner_results = ner_pipeline(text)

    # Sort entities by their start index in descending order to avoid index shifting
    ner_results = sorted(ner_results, key=lambda x: x["start"], reverse=True)
    
    # Remove locations by replacing them with an empty string
    for entity in ner_results:
        text = text[:entity["start"]] + text[entity["end"]:]  # Slice out the location

    # Remove extra spaces caused by deletion
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def remove_place_names(text):

    cleaned_text = [remove_locations(short_text) for short_text in text]
    return cleaned_text

In [10]:
print(text[1578:1580])

print(remove_place_names(text[1578:1580]))

[' During that time I have had cause to use Durning Hall a number of times', ' It is a vital community space']
['During that time I have had cause to use a number of times', 'It is a vital community space']


### Repeat BertTopic modelling with geographic place names removed 

This uses the same fine-tuned huggingface model fine-tuned to have domain specific knowledge from earlier. 

In [11]:
cleaned_text = remove_place_names(text)
topics, probs = topic_model.fit_transform(cleaned_text)

topic_model.get_topic_info()[:10]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1481,-1_community_and_in_of,"[community, and, in, of, the, to, this, for, i...",[I strongly object this application for the fo...
1,0,449,0_diptp_amazing_it_,"[diptp, amazing, it, , , , , , , ]","[, It's amazing"", DipTP]"
2,1,268,1_green_space_was_more,"[green, space, was, more, were, sold, would, a...",[We need more green space in the development a...
3,2,110,2_height_storeys_10_tall,"[height, storeys, 10, tall, high, too, buildin...",[It's still too tall at 10 storeys and shouldn...
4,3,92,3_sunlight_daylight_light_block,"[sunlight, daylight, light, block, my, will, i...",[3) This will greatly impact the daylight/sunl...
5,4,75,4_subsequently_flow_main_substantial,"[subsequently, flow, main, substantial, increa...",[This will have a substantial impact for resid...
6,5,64,5_school_nursery_children_next,"[school, nursery, children, next, dust, noise,...","[The area of , had been in construction for se..."
7,6,62,6_community_space_reduction_use,"[community, space, reduction, use, less, 90, 1...",[It appears that there will be less than 10% o...
8,7,59,7_design_heritage_victorian_architecture,"[design, heritage, victorian, architecture, ch...","[First, the design is unsightly and not in kee..."
9,8,57,8_general_amenity_estate_create,"[general, amenity, estate, create, likely, cau...","[This will create noise pollution, congestion ..."


In [12]:
topic_model.visualize_topics()

In [13]:
topic_model.visualize_heatmap()

In [ ]:
# Hierarchical topics
linkage_function = lambda x: sch.linkage(x, 'single', optimal_ordering=True)
hierarchical_topics = topic_model.hierarchical_topics(cleaned_text, linkage_function=linkage_function)

100%|██████████| 104/104 [00:00<00:00, 626.08it/s]


In [16]:
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

## Different types of topic modelling: Latent Dirichlet Allocation

Latent dirichlet allocation uses bag of words to assign topics - so is not sensitive to the sequence of words, or the way sentences or comments are constructed. 

We start with some additional pre-processing, wherby we remove stop words (if, the, then...), remove all punctuation (.,:...), and also lemmatise (churches -> church, ...).

In [ ]:
# use nltk to remove stopwords, punctuation and lemmatize the text
stop = set(stopwords.words('english'))
exclude = set(string.punctuation)
lemma = WordNetLemmatizer()

def clean(doc):
    stop_free = " ".join([i for i in doc.lower().split() if i not in stop])
    punc_free = ''.join(ch for ch in stop_free if ch not in exclude)
    normalized = " ".join(lemma.lemmatize(word) for word in punc_free.split())
    return normalized

text_clean = [clean(doc).split() for doc in cleaned_text]

In [18]:
text[1578:1581]

[' During that time I have had cause to use Durning Hall a number of times',
 ' It is a vital community space',
 ' A number of campaigns have filled its main hall over the years']

In [19]:
text_clean[1578:1581]

[['time', 'cause', 'use', 'number', 'time'],
 ['vital', 'community', 'space'],
 ['number', 'campaign', 'filled', 'main', 'hall', 'year']]

Convert the imput to a term-document matrix. This is a matrix which counts the occurrence of every term in each document and normalises the counts to create a matrix of values which can be used for LSA or LDA. 

In [ ]:
# Create document-term matrix
dictionary = corpora.Dictionary(text_clean)
doc_term_matrix = [dictionary.doc2bow(doc) for doc in text_clean] 

# Convert sparse to dense format
dense_matrix = [[tup[1] for tup in dictionary.doc2bow(doc)] for doc in text_clean]

In [ ]:
# LDA model 
lda = LdaModel(doc_term_matrix, num_topics=10, id2word = dictionary, passes=50)

[(0, '0.039*"traffic" + 0.038*"development" + 0.021*"construction" + 0.020*"safety" + 0.020*"health"'), (1, '0.036*"building" + 0.025*"new" + 0.024*"object" + 0.017*"development" + 0.015*"area"'), (2, '0.068*"space" + 0.045*"community" + 0.023*"green" + 0.017*"would" + 0.015*"area"'), (3, '0.036*"water" + 0.027*"proposal" + 0.024*"planning" + 0.024*"right" + 0.023*"allow"'), (4, '0.029*"plan" + 0.021*"development" + 0.019*"1" + 0.018*"environmental" + 0.018*"concern"'), (5, '0.036*"construction" + 0.030*"planning" + 0.030*"noise" + 0.030*"school" + 0.021*"building"'), (6, '0.035*"parking" + 0.034*"resident" + 0.028*"additional" + 0.023*"road" + 0.022*"estate"'), (7, '0.027*"resident" + 0.024*"impact" + 0.021*"increase" + 0.019*"proposed" + 0.019*"air"'), (8, '0.026*"child" + 0.017*"class" + 0.013*"community" + 0.013*"school" + 0.010*"activity"'), (9, '0.027*"existing" + 0.023*"building" + 0.019*"development" + 0.018*"area" + 0.016*"currently"')]


In [24]:
# print LDA topics as list 

lda_topics = lda.print_topics(num_topics=10, num_words=7)
lda_topics = [topic[1] for topic in lda_topics]
lda_topics = [topic.split('"') for topic in lda_topics]
lda_topics = [[word for word in topic if word.isalpha()] for topic in lda_topics]

for i, topic in enumerate(lda_topics):
    print(f"Topic {i}: {topic}")

Topic 0: ['traffic', 'development', 'construction', 'safety', 'health', 'movement', 'meter']
Topic 1: ['building', 'new', 'object', 'development', 'area', 'would', 'flat']
Topic 2: ['space', 'community', 'green', 'would', 'area', 'group', 'need']
Topic 3: ['water', 'proposal', 'planning', 'right', 'allow', 'site', 'way']
Topic 4: ['plan', 'development', 'environmental', 'concern', 'access', 'issue']
Topic 5: ['construction', 'planning', 'noise', 'school', 'building', 'application']
Topic 6: ['parking', 'resident', 'additional', 'road', 'estate', 'already', 'amenity']
Topic 7: ['resident', 'impact', 'increase', 'proposed', 'air', 'pollution', 'quality']
Topic 8: ['child', 'class', 'community', 'school', 'activity', 'hall', 'need']
Topic 9: ['existing', 'building', 'development', 'area', 'currently', 'still', 'property']
